# 03 — Topic modelling (LDA)

Cluster the residual (non-government, non-candidate) ad bodies into latent topics so we can:

- Filter out commercial/spam topics (fashion, fitness apps, retail) from the political-adjacent subset.
- Surface what *kinds* of political messaging are circulating outside party/candidate channels — climate, cost-of-living, Voice, housing, etc.
- Layer topic labels into the v3 parquet for cross-tabbing with sentiment (notebook 04) and spend/impressions.

Approach: fit a single LDA model on the full residual corpus, eyeball the top words per topic, hand-label each topic in a CSV, join the labels back to the corpus.

## 1. Preprocessing

Load v2 parquet, filter to one row per ad (`ad_seq_no = 1`) and non-classified (`match_type IS NULL`). Extract the first creative body, tokenise with `RegexTokenizer`, drop stop words (English defaults + domain-specific noise). Vectorise word counts with `CountVectorizer` — raw counts, not TF-IDF, since LDA expects integer term frequencies.

Cache the vectorised features so subsequent LDA fits (at different `k`) don't re-run preprocessing.

### 1.1 Spark session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_contains, col, coalesce, concat_ws, expr, length, lit,
)

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


26/05/11 06:39:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 1.2 Paths

In [2]:
V2_PATH           = '/user/s3348393/main/preprocessing/v2/parquet'
INTERMEDIATE_PATH = '/user/s3348393/main/preprocessing/v3/intermediate_parquet'  # corpus + topic_id
V3_PATH           = '/user/s3348393/main/preprocessing/v3/parquet'               # final with labels

TOPIC_TERMS_CSV   = '../data/topic_terms.csv'    # output: top words per topic (for human review)
TOPIC_LABELS_CSV  = '../data/topic_labels.csv'   # input: human-edited labels

### 1.3 Load v2 and filter to the residual corpus

Keep one row per ad (`ad_seq_no = 1`) and only ads that:

- didn't classify as candidate/party/government (`match_type IS NULL`)
- are English-language (`array_contains('languages', 'en')`) — drops non-English clusters
- aren't from a byline we've explicitly tagged as commercial/non-political (`COMMERCIAL_BYLINES`) — iteratively grown as LDA surfaces noise topics

Body-text extraction is more inclusive than just `creative_bodies[0]`:

1. `first_non_empty(arr)` returns the **first non-null, non-empty** element of an array column — recovers ads where the first creative-body entry is null but a later one is real (multi-variant ads), or where the singular `ad_creative_body` was null but other text fields are populated.
2. Concatenate `creative_bodies` + `creative_link_descs` + `creative_link_titles` into a single document — more text = better LDA signal, and ads with no body but a populated link description ("Sign the petition", "Donate now") still contribute.
3. Drop ads with no text in any of these fields — LDA can't process empty documents.

`creative_link_captions` is skipped because it's usually just a domain name (low signal, noise).

In [3]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


# Bylines we've identified as clearly non-political (commercial, recruitment, etc.)
# and want excluded from the LDA corpus. Iteratively grown — add bylines here when
# LDA surfaces them as dominating a noise/commercial topic.
COMMERCIAL_BYLINES = {
    'Access',                            # Indigenous Employment Australia — job listings
    'Streamotion Pty Ltd',               # Kayo / Binge sports + entertainment streaming
    'SBS Australia',                     # SBS On Demand streaming promotions
    'SBS Arabic24',                      # SBS language-stream marketing
    'SBS Mandarin中文普通话',            # SBS language-stream marketing
    'The Squiz',                         # paid news newsletter
    'Hair Cooki'                           #hair care ad
}


corpus = df.filter(
        (col('ad_seq_no') == 1) &
        col('match_type').isNull() &
        # Language: keep ads where languages is null (untagged — ~96% of residual)
        # OR explicitly tagged as English. Drops the small confirmed-non-English tail.
        (col('languages').isNull() | array_contains('languages', 'en')) &
        ~col('bylines').isin(list(COMMERCIAL_BYLINES))
    ) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

All v2 rows:     5796491


Residual corpus: 96286
+---------------------------------+---------------------------------+--------------------------------------------------------------------------------+
|                        page_name|                          bylines|                                                                            body|
+---------------------------------+---------------------------------+--------------------------------------------------------------------------------+
|    Australian Ethical Investment|    Australian Ethical Investment|Creating sustainable impact or getting top returns? Por que no los dos?! 🤷\n...|
|Australian Automobile Association|Australian Automobile Association|We’re all starting to move again, but we need to get to our destination effic...|
|     Greenpeace Australia Pacific|     Greenpeace Australia Pacific|DID YOU KNOW that if global temperatures warm to 🔥 2.0 degrees above pre-ind...|
+---------------------------------+---------------------------------+----

### 1.4 Stop words

English defaults plus a small list of domain-specific noise: URL fragments, generic call-to-action words. Keep this list deliberately short — `minDF` and `maxDF` in `CountVectorizer` will handle most of the frequency-based filtering automatically. Iterate after the first LDA fit if specific tokens are dominating topics with no signal.

In [4]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # contraction fragments surviving minTokenLength=2
    're', 've', 'll',
    # generic fillers (high frequency, low topic-discrimination value)
    'help', 'time', 'like', 'need', 'make', 'take', 'people',
    'year', 'years', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
    # generic CTA (kept short — don't strip 'petition', 'donate', 'sign', etc.
    # since those carry topic signal)
    'click', 'learn',
]

print('Stop-words list size:', len(stop_words))

Stop-words list size: 215


### 1.5 Preprocessing pipeline

Three stages: `RegexTokenizer` (split on `\W+`, lowercase, drop tokens shorter than 2 chars) → `StopWordsRemover` (English defaults + domain noise) → `CountVectorizer` (vocab≤5,000, term must appear in ≥100 ads, term must appear in ≤30% of ads).

`minDF=100` is tighter than the conventional default — it compresses the long-tail vocabulary used by individual small commercial advertisers, forcing LDA to cluster around vocabulary that's actually shared across the political-advocacy ecosystem. Niche-but-real political terms (`woodside`, `quoll`, `uyghur`) should still clear the threshold; one-off commercial product names won't.

Wrap in a `Pipeline` so we can `.fit().transform()` in one go and have a single fitted artefact to introspect afterwards.

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000,
    minDF=100,    # term must appear in ≥100 ads — bumped from 50 to compress long-tail vocab
    maxDF=0.3,    # term appearing in >30% of ads is dropped (auto stop-word filter)
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

### 1.6 Fit, transform, cache

Fit the pipeline once. The output `features_df` carries every original column plus `raw_tokens`, `tokens`, and `features` (the sparse count vector LDA consumes). Cache it so the k-sweep in section 2 doesn't re-run preprocessing per `k`.

The `.count()` call forces Spark to actually materialise the cache — without it, the cache is registered lazily and nothing happens until something else triggers an action.

In [6]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

print('Cached rows:    ', features_df.count())

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
print(vocab[:30])

26/05/11 06:40:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Cached rows:     96286
Vocabulary size: 4318

Top 30 vocabulary terms (most frequent first):
['sign', 'australia', 'climate', 'government', 'support', 'community', 'petition', 'australian', 'change', 'vote', 'world', 'protect', 'women', 'future', 'action', 'local', 'free', 'join', 'election', 'stop', 'share', 'council', 'woodside', 'children', 'labor', 'life', 'donate', 'every', 'energy', 'gas']


## 2. Explore `k` on a sample

Fit LDA at several `k` values (5, 10, 15, 20) on a 10% sample of the cached features. Print top-12 words per topic for each `k`. Eyeball the printouts to pick a `k` where topics are distinct and each list reads as a coherent theme.

Fix `seed=42` so comparing `k=10` vs `k=15` isn't muddled by random init differences. Cheap and disposable — no parquet writes from this section.

In [7]:
from pyspark.ml.clustering import LDA

# 10% sample of the cached features. Cache the sample too — the four LDA fits
# below will scan it repeatedly. seed=42 keeps the sample composition stable.
sample_df = features_df.sample(0.1, seed=42).cache()
print(f'Sample size: {sample_df.count():,}')

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20]:
    print(f'\n=== k = {k} ===')
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=12).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

Sample size: 9,663

=== k = 5 ===


26/05/11 06:40:29 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: community climate sign australia local woodside council government share tell gas future
  Topic  1: support women community violence council club young work residents family pay living
  Topic  2: australia sign government petition support australian climate protect change world children vote
  Topic  3: energy power sign climate electricity renewable change coal ocean care agl world
  Topic  4: early super learning future free childcare sign life protect women support gift

=== k = 10 ===


26/05/11 06:40:41 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: woodside sign australia gas tell energy share climate enough clean solar project
  Topic  1: women clothes demand paid brands wage big pay living nike club australian
  Topic  2: australia abc sign petition medicines cost australian save government join world prescription
  Topic  3: electricity energy green renewable switch switching coffee bills power pain level good
  Topic  4: vaccine reef climb christmas australians choice freedom sign choose free vaxxed show
  Topic  5: climate community vote australia future change council local government action support health
  Topic  6: sign women support government petition children early donate families super australian australia
  Topic  7: support quiz power freedom government labor energy first zero minute raise feast
  Topic  8: ocean sign treaty oceans petition global support life police 2022 march demand
  Topic  9: protect nature species laws sign habitat forest save australia stop petition strong

=== k = 15 ===


26/05/11 06:40:51 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: climate action australia clean transport australians energy crisis change safer toyota reduce
  Topic  1: women clothes paid brands wage demand living nike big sign forestry pay
  Topic  2: medicines australia cost prescription festival bank live join visit program book feast
  Topic  3: electricity green switch switching pain provider guide greener abortion climate deforestation pledge
  Topic  4: vaccine women reef rights childcare climb fair australians sign learning freedom choose
  Topic  5: change school future gift climate education planet native protect kids support christmas
  Topic  6: sign women support donate early children plastic violence use learning families single
  Topic  7: quiz minute first rights support campaign research raise money really day human
  Topic  8: life refugees donation provide support death asylum access work donate case text
  Topic  9: nature protect species laws habitat donate save strong sign destruction world forest
  Topic 10: comm

26/05/11 06:41:01 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: toyota transport clean australia car electric vehicles australians climate cleaner deserve renters
  Topic  1: women clothes paid brands wage big nike living demand pledge hour pay
  Topic  2: medicines australia cost prescription bank festival live program visit join book choose
  Topic  3: deforestation switching electricity pain european pledge greener makers science citizen impact donate
  Topic  4: women learning rights early childcare freedom vaccine human quiz fair equality sign
  Topic  5: education future children child gift change school kids support life protect every
  Topic  6: donate support children women early donation violence provide families educators refugees food
  Topic  7: day really first nations history support significant australian find many loss vision
  Topic  8: life death rocky text case free send work message visa legal won
  Topic  9: protect nature species habitat donate forest rainforest laws ukraine destruction national sign
  Topic 10: c

DataFrame[id: string, page_id: string, page_name: string, snapshot_date: date, ad_creation_date: date, ad_delivery_start_date: date, ad_delivery_stop_date: date, creative_bodies: array<string>, creative_link_captions: array<string>, creative_link_descs: array<string>, creative_link_titles: array<string>, currency: string, languages: array<string>, publisher_platforms: array<string>, demographic_distribution: array<struct<age:string,gender:string,percentage:string>>, delivery_by_region: array<struct<percentage:string,region:string>>, ad_snapshot_url: string, bylines: string, spend_lower_bound: bigint, spend_upper_bound: bigint, spend_mid: double, impressions_lower_bound: bigint, impressions_upper_bound: bigint, impressions_mid: double, audience_size_lower_bound: bigint, audience_size_upper_bound: bigint, audience_size_mid: double, ad_seq_no: int, match_type: string, political_party: string, body: string, raw_tokens: array<string>, tokens: array<string>, features: vector]

## 3. Final fit on full corpus

Refit LDA at `k=20` on the full cached features (no sampling). Transform the corpus to attach `topicDistribution` (length-20 vector) and `topic_id` (argmax) to every ad. Then persist two artefacts:

- **Intermediate parquet** — full corpus + LDA columns. Expensive to recompute (LDA on 50k docs at k=20 takes ~30–60s); write once so the labelling round-trip doesn't refit.
- **`data/topic_terms.csv`** — one row per topic with `top_terms` (top 15 words) and `top_bylines` (top 5 advertisers by ad count). The bylines column is the key labelling aid: combined with the term list, you can usually tell whether a topic is e.g. *Climate Council reef campaign* (top byline: Climate Council) vs *Greenpeace anti-Woodside* (top byline: Greenpeace) vs *commercial residue* (top byline: a brand) — much faster than guessing from terms alone.

The CSV also has an empty `label` column for you to fill in. Save as `data/topic_labels.csv` when done.

In [8]:
from pyspark.ml.clustering import LDA
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

K = 20

# Fit LDA at k=K on the full cached features. seed=42 for reproducibility.
lda = LDA(featuresCol='features', k=K, maxIter=20, seed=42)
lda_model = lda.fit(features_df)

# Transform: attach topicDistribution and dominant topic_id to every ad.
classified = lda_model.transform(features_df) \
    .withColumn('topic_array', vector_to_array('topicDistribution')) \
    .withColumn('topic_id', expr('array_position(topic_array, array_max(topic_array)) - 1'))

# Vocabulary for term-index -> word translation.
vocab = prep_model.stages[-1].vocabulary

# Top 15 terms per topic from describeTopics.
topics_rows = lda_model.describeTopics(maxTermsPerTopic=15).collect()
topic_terms = {row.topic: [vocab[i] for i in row.termIndices] for row in topics_rows}

# Top 5 bylines per topic — Spark window function over (topic_id, bylines, count).
w = Window.partitionBy('topic_id').orderBy(desc('count'))
top_bylines_rows = classified.filter(col('bylines').isNotNull()) \
    .groupBy('topic_id', 'bylines').count() \
    .withColumn('rank', row_number().over(w)) \
    .filter(col('rank') <= 5) \
    .orderBy('topic_id', 'rank') \
    .collect()

top_bylines = {}
for row in top_bylines_rows:
    top_bylines.setdefault(row.topic_id, []).append(row.bylines)

# Print to console — easy to scan while you draft labels.
print(f'k = {K}\n')
for tid in range(K):
    words = ' '.join(topic_terms.get(tid, []))
    bls   = ' | '.join(top_bylines.get(tid, []))
    print(f'Topic {tid:>2}:')
    print(f'  Terms:   {words}')
    print(f'  Bylines: {bls}')
    print()

26/05/11 06:41:12 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


k = 20

Topic  0:
  Terms:   climate sign oceans action change ocean global australia treaty crisis petition world act leaders clean
  Bylines: Greenpeace Australia Pacific | The Climate Council | Australian Conservation Foundation | Comms Declare Inc | Oxfam Australia

Topic  1:
  Terms:   violence child children domestic women abuse support stop girls sex sexual end name safe add
  Bylines: St Vincent de Paul Society in Australia | Destiny Rescue | Save the Children Australia | Plan International Australia | NATIONAL ZAKAT FOUNDATION INCORPORATED

Topic  2:
  Terms:   australia woodside project environment nature laws save join gas festival tell wildlife message habitat sign
  Bylines: Festival of Dangerous Ideas | Greenpeace Australia Pacific | Australian Conservation Foundation | The Wilderness Society | Bank Australia

Topic  3:
  Terms:   energy coal electricity renewable power agl australia green clean climate future zero switch sustainability genus
  Bylines: Greenpeace Austral

In [9]:
import pandas as pd

# Write topic_terms.csv — one row per topic with top terms + top bylines + empty
# label column for the human to fill in.
topic_terms_pdf = pd.DataFrame([
    {
        'topic_id':    tid,
        'top_terms':   ' '.join(topic_terms.get(tid, [])),
        'top_bylines': '|'.join(top_bylines.get(tid, [])),
        'label':       '',
    }
    for tid in range(K)
])
topic_terms_pdf.to_csv(TOPIC_TERMS_CSV, index=False)
print(f'Wrote {TOPIC_TERMS_CSV}')

# Write intermediate parquet — corpus + topic_id + topicDistribution.
# Drop the large intermediate columns (token arrays, sparse vectors) so the
# parquet is just the v2 corpus plus the LDA-derived columns.
intermediate = classified.drop('raw_tokens', 'tokens', 'features', 'topic_array')
spark.conf.set('spark.sql.parquet.output.committer.class',
               'org.apache.parquet.hadoop.ParquetOutputCommitter')
intermediate.write \
    .option('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .parquet(INTERMEDIATE_PATH, mode='overwrite')
print(f'Wrote {INTERMEDIATE_PATH}')

Wrote ../data/topic_terms.csv


26/05/11 06:41:31 WARN DAGScheduler: Broadcasting large task binary with size 1127.1 KiB


Wrote /user/s3348393/main/preprocessing/v3/intermediate_parquet


## 4. Manual labelling (out-of-notebook)

Open `data/topic_terms.csv` in a spreadsheet, add a `label` column with human-readable names (e.g. `climate`, `cost_of_living`, `voice_referendum`, `commercial_retail`, `noise`). Save as `data/topic_labels.csv`.

Topics that look like commercial noise (fashion brands, fitness app keywords, etc.) get labels like `commercial_*` or `noise` — these are the categories notebook 04 / later filters will drop.

## 5. Join labels and filter to the political-adjacent corpus

Two jobs: join the hand-edited labels onto the corpus, then apply a filter that produces the v3 parquet [notebook 05](05_temporal.ipynb) will analyse.

**Filter rule**:

- **KEEP** `match_type ∈ {candidate, party_org}` — registered party advertising (every candidate, every party central office).
- **KEEP** `match_type IS NULL AND category ≠ 'noise'` — political-adjacent advocacy that survived both the LDA corpus filter (English, non-commercial-byline, non-empty body) AND the hand-labelling round (not tagged `noise`).
- **DROP** `match_type = 'government'` — government department advertising, out of scope for the political-spending analysis.
- **DROP** residual ads with `category = 'noise'` (LDA-surfaced commercial / streaming / mixed).
- **DROP** residual ads with no topic at all (empty-body, non-English, commercial-byline filters in cell 7).

Result: *all 2022 Australian non-government political-adjacent FB ads*, in one parquet, partitioned by `category`, ready for the election-only-vs-ongoing analysis.

Before writing, sanity-check the joined data:

- **Distribution** — `match_type × category` cross-tab should produce expected counts.
- **False-positive review** — top-spend ads in each kept category should look like genuine political content.
- **False-negative review** — top advertisers from dropped (noise) topics should look genuinely commercial / non-political.

### 5.1 Join labels and topic info onto v2

Three inputs combined:

1. `v2` parquet (one row per ad via `ad_seq_no = 1`) — every ad with `match_type` and `political_party`.
2. `intermediate` parquet (residual subset only, with `topic_id` + `topicDistribution`) — produced by section 3.
3. `topic_labels.csv` — hand-edited labels with `topic_label` and `category`.

Left-join so every v2 row gets the LDA columns *if* it was in the residual corpus, otherwise null. Cache the result — every cell below scans it.

In [ ]:
from pyspark.sql.functions import broadcast

# v2: full classified corpus, one row per ad
v2 = spark.read.parquet(V2_PATH).filter(col('ad_seq_no') == 1)

# Residual subset with LDA topic info (from section 3's intermediate parquet)
intermediate = spark.read.parquet(INTERMEDIATE_PATH) \
    .select('id', 'topic_id', 'topicDistribution', 'body')

# Hand-edited labels: topic_id -> (topic_label, category)
labels = spark.read.csv(TOPIC_LABELS_CSV, header=True) \
    .selectExpr('cast(topic_id as int) as topic_id',
                'label as topic_label',
                'category')

# Left-join: every ad gets topic info IF it was in the residual corpus, else null.
classified = v2 \
    .join(intermediate, 'id', 'left') \
    .join(broadcast(labels), 'topic_id', 'left') \
    .cache()

print(f'All ads (ad_seq_no=1): {classified.count():,}')
print(f'  with topic info:    {classified.filter(col("topic_id").isNotNull()).count():,}')
print(f'  without topic info: {classified.filter(col("topic_id").isNull()).count():,}')

### 5.2 Distribution sanity check

Confirm the join produced sensible counts. Non-residual rows (candidate / party_org / government) should have null `category`; residual rows (`match_type IS NULL`) should split across the labelled categories.

In [ ]:
# match_type × category cross-tab — confirms the join produced expected counts.
# Non-residual rows (candidate, party_org, government) should have null category;
# residual rows (match_type IS NULL) should be spread across the labelled categories.
print('match_type × category:')
classified.groupBy('match_type', 'category').count() \
    .orderBy('match_type', 'category') \
    .show(50, truncate=False)

# Belt-and-braces: did any topic_id end up unmatched in the labels CSV?
unmatched = classified.filter(col('topic_id').isNotNull() & col('topic_label').isNull()).count()
print(f'Rows with topic_id but no topic_label (should be 0): {unmatched}')

### 5.3 Apply the political-corpus filter

Drop government rows, residual noise-labelled rows, and residual rows that didn't make the LDA corpus (no topic). What survives is the analysis-ready political-adjacent corpus.

In [ ]:
before = classified.count()

v3_candidate = classified.filter(
    col('match_type').isin('candidate', 'party_org')
    | (col('match_type').isNull() & col('category').isNotNull() & (col('category') != 'noise'))
)
v3_candidate.cache()
after = v3_candidate.count()

print(f'Before filter: {before:,}')
print(f'After filter:  {after:,}')
print(f'Dropped:       {before - after:,}  ({100 * (before - after) / before:.1f}%)')

# What survived, broken down
print('\nKept rows by match_type × category:')
v3_candidate.groupBy('match_type', 'category').count() \
    .orderBy('match_type', 'category') \
    .show(50, truncate=False)

### 5.4 Validation — false positives and false negatives

**False positives** (we INCLUDED these — do they look genuinely political?): top-spend ads from each kept category. If a category surfaces a commercial advertiser at the top, the LDA labelling needs another pass.

**False negatives** (we EXCLUDED these — were any actually political?): top advertisers from the dropped (noise) topics, and top advertisers from residual rows that didn't make the LDA corpus at all (no topic_id). If a real political advertiser surfaces, that topic was mislabelled or that byline was over-filtered upstream.

In [ ]:
from pyspark.sql.functions import desc

# --- False positives: are the top-spend ads in each kept category genuinely political? ---
print('=== FP check: top-spend per kept category ===')
for cat in ['climate', 'humanitarian_rights', 'political_advocacy', 'cost_of_living']:
    print(f'\n--- {cat} ---')
    v3_candidate.filter((col('category') == cat) & col('spend_mid').isNotNull()) \
        .orderBy(desc('spend_mid')) \
        .select('page_name', 'bylines', 'topic_label', 'spend_mid') \
        .show(5, truncate=60)

# Also check the party/candidate matches — top-spend by party
print('\n=== FP check: top-spend candidate + party_org ads ===')
v3_candidate.filter(col('match_type').isin('candidate', 'party_org') & col('spend_mid').isNotNull()) \
    .orderBy(desc('spend_mid')) \
    .select('page_name', 'bylines', 'political_party', 'match_type', 'spend_mid') \
    .show(10, truncate=60)


# --- False negatives: are any of the dropped (noise) topics' top bylines actually political? ---
print('\n=== FN check: top bylines in DROPPED noise-labelled topics ===')
classified.filter(col('category') == 'noise') \
    .groupBy('topic_label', 'bylines').count() \
    .orderBy(desc('count')) \
    .show(20, truncate=80)

# --- False negatives: top bylines in residual that had NO topic (dropped by cell-7 corpus filter) ---
print('\n=== FN check: top bylines in residual with NO topic_id (empty body / non-English / commercial bylines) ===')
classified.filter(col('match_type').isNull() & col('topic_id').isNull() & col('bylines').isNotNull()) \
    .groupBy('bylines').count() \
    .orderBy(desc('count')) \
    .show(20, truncate=80)

### 5.5 Write v3 + round-trip check

Drop intermediate columns we don't need downstream (`body`, `topicDistribution`) — keep the labels and v2 metadata. Write partitioned by `category` so notebook 05 can read subsets selectively.

In [ ]:
intermediate_cols = ['body', 'topicDistribution']
v3_out = v3_candidate.drop(*[c for c in intermediate_cols if c in v3_candidate.columns])

spark.conf.set('spark.sql.parquet.output.committer.class',
               'org.apache.parquet.hadoop.ParquetOutputCommitter')
v3_out.write \
    .option('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .partitionBy('category') \
    .parquet(V3_PATH, mode='overwrite')

print(f'Wrote {V3_PATH}')

# Round-trip check
rt = spark.read.parquet(V3_PATH)
print(f'\nRound-trip rows: {rt.count():,}')
rt.printSchema()